# Get Walgreens locations

#### Load Python tools and Jupyter config

In [1]:
import us 
import json
import black
import requests
import pandas as pd
import jupyter_black
import altair as alt
import geopandas as gpd
from bs4 import BeautifulSoup
from vega_datasets import data
from tqdm.notebook import tqdm, trange

In [2]:
jupyter_black.load()
pd.options.display.max_columns = 100
pd.options.display.max_rows = 1000
pd.options.display.max_colwidth = None
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [3]:
place = "walgreens"
place_formal = "Walgreens"
color = "#a32a33"

---

## Scrape

#### Headers for our request

In [4]:
headers = {
    "authority": "www.walgreens.com",
    "accept": "application/json, text/plain, */*",
    "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
}

#### Import the county's largest ZIP Codes and ensure they have five digits

In [5]:
zips = (
    pd.read_json("../_reference/data/zip_code_demographics_esri.json")
    .query("population > 1000")
    .sort_values("population", ascending=False)
    .reset_index(drop=True)
)
zips["zipcode"] = zips["zipcode"].astype(str).str.zfill(5)

In [6]:
len(zips)

23108

#### Loop through sample of ZIP Codes to request stores within a 100 miles

In [7]:
response_list = []

sample = zips.copy()

for index, row in tqdm(sample.iterrows(), total=len(sample)):
    params = {
        "requestor": "search",
    }

    json_data = {
        "r": "50",
        "requestType": "dotcom",
        "s": "50",
        "q": f'{row["zipcode"]}',
        "zip": f'{row["zipcode"]}',
    }

    response = requests.post(
        "https://www.walgreens.com/locator/v1/stores/search",
        params=params,
        headers=headers,
        json=json_data,
    )

    # Check if "results" key exists in the response
    if "results" in response.json():
        for r in response.json()["results"]:
            store_number = r["storeNumber"]
            street_address = r["store"]["address"]["street"].title()
            city = r["store"]["address"]["city"].title()
            state = r["store"]["address"]["state"]
            zip_code = r["store"]["address"]["zip"]
            phone = r["store"]["phone"]["number"]
            latitude = r["latitude"]
            longitude = r["longitude"]

            response_dict = {
                "store_number": store_number,
                "street": street_address,
                "city": city,
                "state": state,
                "zip": zip_code,
                "latitude": latitude,
                "longitude": longitude,
            }
            response_list.append(response_dict)
    else:
        print(f"No results for {row['zipcode']}.")

  0%|          | 0/23108 [00:00<?, ?it/s]

No results for 96720.
No results for 58801.
No results for 17701.
No results for 96740.
No results for 59718.
No results for 58503.
No results for 59715.
No results for 58701.
No results for 58601.
No results for 57401.
No results for 58501.
No results for 58504.
No results for 99801.
No results for 58554.
No results for 97603.
No results for 89801.
No results for 97078.
No results for 58703.
No results for 92225.
No results for 59714.
No results for 95531.
No results for 97601.
No results for 79065.
No results for 96749.
No results for 17837.
No results for 85607.
No results for 96746.
No results for 17745.
No results for 83001.
No results for 58401.


SSLError: HTTPSConnectionPool(host='www.walgreens.com', port=443): Max retries exceeded with url: /locator/v1/stores/search?requestor=search (Caused by SSLError(SSLError(1, '[SSL: DECRYPTION_FAILED_OR_BAD_RECORD_MAC] decryption failed or bad record mac (_ssl.c:2633)')))

#### Read list of results to a dataframe and remove any duplicates from adjacent ZIP Codes

In [ ]:
df = pd.DataFrame(response_list).drop_duplicates(subset="store_number")

#### How many are left? 

In [ ]:
len(df)

#### Create a mapping of state abbreviations to full state names using the us library

In [ ]:
state_mapping = {state.abbr: state.name for state in us.states.STATES}

#### New column of full state names based on abbreviations

In [ ]:
df["state_name"] = df["state"].map(state_mapping)

#### Make sure our brand gets in the dataframe

In [ ]:
df["brand"] = place_formal

---

## Geography

#### Make it a geodataframe

In [ ]:
df_geo = df.copy()

In [ ]:
gdf = gpd.GeoDataFrame(
    df_geo, geometry=gpd.points_from_xy(df_geo.longitude, df_geo.latitude)
)

---

## Maps

#### US states background

In [ ]:
background = (
    alt.Chart(alt.topo_feature(data.us_10m.url, feature="states"))
    .mark_geoshape(fill="#e9e9e9", stroke="white")
    .properties(width=800, height=500, title=f"{place_formal} locations")
    .project("albersUsa")
)

#### Location points map

In [ ]:
points = (
    alt.Chart(gdf)
    .mark_circle(size=5, color=color)
    .encode(
        longitude="longitude:Q",
        latitude="latitude:Q",
    )
)

point_map = background + points
point_map.configure_view(stroke=None)

#### Location proportional symbols map

In [ ]:
symbols = (
    alt.Chart(gdf)
    .transform_aggregate(
        latitude="mean(latitude)",
        longitude="mean(longitude)",
        count="count()",
        groupby=["state"],
    )
    .mark_circle()
    .encode(
        longitude="longitude:Q",
        latitude="latitude:Q",
        size=alt.Size("count:Q", title="Count by state"),
        color=alt.value(color),
        tooltip=["state:N", "count:Q"],
    )
    .properties(
        title=f"Number of {place_formal} in US, by average lon/lat of locations"
    )
)

symbol_map = background + symbols
symbol_map.configure_view(stroke=None)

---

## Exports

#### JSON

In [ ]:
df.to_json(
    f"data/processed/{place.lower().replace(' ', '_')}_locations.json",
    indent=4,
    orient="records",
)

#### CSV

In [ ]:
df.to_csv(
    f"data/processed/{place.lower().replace(' ', '_')}_locations.csv", index=False
)

#### GeoJSON

In [ ]:
gdf.to_file(
    f"data/processed/{place.lower().replace(' ', '_')}_locations.geojson",
    driver="GeoJSON",
)